# Draft Simulator

Simulates a 10-team snake draft: our agent (using the trained model's
projections) vs. 9 opponents who draft strictly by ADP. Evaluated on the
2024 season -- the untouched test season from the modeling notebook, so
this is a genuinely fair, out-of-sample check.

**Scope note**: the whole project is restricted to QB/RB/WR/TE (no K/DST --
decided back in the EDA phase, since DST scoring needs a separate
team-level pipeline we never built, and kickers are low-signal). The
simulated league reflects that same scope: rosters only have skill-position
slots.


In [110]:
import sys
sys.path.append('../src')

import json
import random

import pandas as pd
import numpy as np
import xgboost as xgb

from build_features import build_features, get_feature_columns, SKILL_POSITIONS

pd.set_option('display.max_columns', 100)


## Step 1: build the 2024 player pool

Each player needs: model projection (from the trained model, using features
built from their PRE-2024 data -- exactly what the model saw at test time),
ADP (what the opponent bots draft by), and actual 2024 outcome (to score
the simulated season afterward).


In [111]:
seasonal = pd.read_parquet('../data/raw/seasonal_player_stats.parquet')
rosters = pd.read_parquet('../data/raw/seasonal_rosters.parquet')
injuries = pd.read_parquet('../data/raw/injuries.parquet')
adp_with_outcomes = pd.read_parquet('../data/processed/adp_with_outcomes.parquet')

features = build_features(seasonal, rosters, injuries)
features_encoded = pd.get_dummies(features, columns=['position'], prefix='pos')

with open('../models/feature_columns.json') as f:
    feature_cols = json.load(f)

model = xgb.XGBRegressor()
model.load_model('../models/xgb_projection_model.json')

season_2024 = features_encoded[features_encoded['season'] == 2024].copy()
season_2024['model_pred'] = model.predict(season_2024[feature_cols])

print(f"2024 player pool (pre-ADP-merge): {season_2024.shape[0]} players")


2024 player pool (pre-ADP-merge): 569 players


In [112]:
# Bring position back (it got one-hot encoded), plus ADP and actual outcome.
position_lookup = features[features['season'] == 2024][['player_id', 'position']]
adp_2024 = adp_with_outcomes[adp_with_outcomes['season'] == 2024][['player_id', 'player_name', 'adp_avg']]

player_pool = season_2024[['player_id', 'model_pred', 'fantasy_points_ppr']].rename(
    columns={'fantasy_points_ppr': 'actual_points'}
)
player_pool = player_pool.merge(position_lookup, on='player_id', how='left')
player_pool = player_pool.merge(adp_2024, on='player_id', how='inner')  # inner: need ADP to be draftable

player_pool = player_pool.sort_values('adp_avg').reset_index(drop=True)

print(f"Final draftable player pool: {player_pool.shape[0]} players")
player_pool.head(15)


Final draftable player pool: 542 players


,player_id,model_pred,actual_points,position,player_name,adp_avg
0,00-0033280,263.611084,47.80,RB,Christian McCaffrey,1.0
1,00-0036358,223.544373,263.40,WR,CeeDee Lamb,2.6
2,00-0033040,235.289337,218.20,WR,Tyreek Hill,3.2
3,00-0038542,218.672531,341.70,RB,Bijan Robinson,5.0
4,00-0038120,280.011200,240.90,RB,Breece Hall,5.4
5,00-0036963,262.420441,316.18,WR,Amon-Ra St. Brown,6.2
6,00-0036900,222.933167,403.00,WR,Ja'Marr Chase,6.6
7,00-0036322,251.414200,317.48,WR,Justin Jefferson,7.0
8,00-0034844,167.677185,355.30,RB,Saquon Barkley,9.2
9,00-0035676,241.148743,216.90,WR,A.J. Brown,10.2


## Step 2: draft mechanics

**Roster** (skill positions only): 1 QB, 2 RB, 2 WR, 1 TE, 1 FLEX (RB/WR/TE)
= 7 starters + 6 bench = 13 rounds/team, 10 teams = 130 players drafted.

**Drafting logic**: best-player-available by ranking, with soft position
caps (max QB:2, RB:5, WR:5, TE:2 per team) so bots don't hoard one position
unrealistically. This is a simplification vs. full slot-by-slot roster
logic, but a standard and reasonable one for this kind of simulation.

**Our agent's ranking**: a blend of model and ADP percentile ranks,
weighted by position based on what the modeling notebook actually found --
leaning toward the model where it earned it (RB/TE), toward ADP where the
market is still clearly stronger (QB/WR).


In [113]:
def compute_agent_scores(pool):
    """Blended ranking for our agent, expressed on the ADP scale (LOWER is
    better) so the model and ADP signals are directly comparable and
    combinable -- fixes an earlier bug where within-position percentiles
    threw away real cross-position scarcity information (e.g. treating a
    TE's 90th percentile as equivalent in value to an RB's 90th percentile,
    when real market ADP never would).

    Approach: for each position, rank players by model_pred. Map each
    player's model-based positional rank to the ADP value of whichever
    player holds that SAME positional rank by real ADP. This lets the
    model re-order players within a position based on its own predictions,
    while inheriting the market's real cross-position value calibration.
    """
    pool = pool.copy()

    for pos in SKILL_POSITIONS:
        pos_mask = pool['position'] == pos
        pos_players = pool[pos_mask]

        model_order = pos_players.sort_values('model_pred', ascending=False)
        adp_order = pos_players.sort_values('adp_avg', ascending=True)['adp_avg'].values

        # model_order's Nth player (by model rank) gets assigned the ADP
        # value belonging to the Nth-ranked player by real ADP.
        model_implied_adp = pd.Series(adp_order[:len(model_order)], index=model_order.index)
        pool.loc[model_implied_adp.index, 'model_implied_adp'] = model_implied_adp

    weight_map = pool['position'].map(MODEL_WEIGHT_BY_POSITION)
    pool['agent_score'] = -(
        weight_map * pool['model_implied_adp'] + (1 - weight_map) * pool['adp_avg']
    )  # negated so higher agent_score = better, consistent with "sort descending" used at pick time

    return pool


## Step 3: the actual draft simulation

One team (`agent_team_index`) drafts using the blended agent score; every
other team drafts strictly by lowest ADP. At each pick: filter to players
not yet drafted and not at their position cap for that team, then take the
single best-ranked remaining player.


In [114]:
def run_draft(pool, agent_team_index, seed=None):
    if seed is not None:
        random.seed(seed)

    pool = compute_agent_scores(pool)
    available = pool.copy()
    rosters = {t: [] for t in range(N_TEAMS)}
    position_counts = {t: {'QB': 0, 'RB': 0, 'WR': 0, 'TE': 0} for t in range(N_TEAMS)}

    pick_order = snake_order(N_TEAMS, TOTAL_ROUNDS)

    for team in pick_order:
        # Filter out positions this team has already capped out on.
        eligible = available[available['position'].map(
            lambda p, t=team: position_counts[t][p] < POSITION_CAPS[p]
        )]
        if eligible.empty:
            eligible = available  # fallback: caps exhausted everywhere, just take best available

        if team == agent_team_index:
            pick = eligible.sort_values('agent_score', ascending=False).iloc[0]
        else:
            pick = eligible.sort_values('adp_avg', ascending=True).iloc[0]

        rosters[team].append(pick)
        position_counts[team][pick['position']] += 1
        available = available[available['player_id'] != pick['player_id']]

    return rosters

# Quick smoke test: run one draft with our agent in slot 0.
test_rosters = run_draft(player_pool, agent_team_index=0, seed=1)
print(f"Team 0 (our agent) drafted {len(test_rosters[0])} players:")
pd.DataFrame(test_rosters[0])[['player_name', 'position', 'adp_avg', 'model_pred', 'actual_points']]


Team 0 (our agent) drafted 13 players:


,player_name,position,adp_avg,model_pred,actual_points
4,Breece Hall,RB,5.4,280.011200,240.9
26,Josh Jacobs,RB,28.2,210.025635,293.1
24,De'Von Achane,RB,27.4,206.297745,299.9
41,Kenneth Walker III,RB,41.4,197.241409,181.2
39,Alvin Kamara,RB,39.6,184.793823,265.3
61,Zay Flowers,WR,63.2,191.170425,209.5
77,Keenan Allen,WR,79.4,224.653290,184.4
81,David Njoku,TE,84.2,148.899872,148.5
82,Jayden Reed,WR,87.2,178.716476,197.0
116,T.J. Hockenson,TE,126.4,195.219116,86.5


## Step 4: scoring a drafted roster

For each team, fill required slots (QB, RB, RB, WR, WR, TE) with that
team's best ACTUAL-points player at each position, then fill FLEX with the
best remaining RB/WR/TE. Sum the starters' actual points -- this is what
that team "scored" for the season. This greedy fill is optimal for this
simple slot structure (exact-position slots get first pick of the best
players, FLEX takes the best leftover).


In [115]:
def score_roster(roster_players):
    roster_df = pd.DataFrame(roster_players)
    used_ids = set()
    starters = []

    for pos, n_slots in [('QB', 1), ('RB', 2), ('WR', 2), ('TE', 1)]:
        pos_players = roster_df[roster_df['position'] == pos].sort_values('actual_points', ascending=False)
        pos_players = pos_players[~pos_players['player_id'].isin(used_ids)]
        picks = pos_players.head(n_slots)
        starters.append(picks)
        used_ids.update(picks['player_id'])

    flex_eligible = roster_df[roster_df['position'].isin(['RB', 'WR', 'TE'])]
    flex_eligible = flex_eligible[~flex_eligible['player_id'].isin(used_ids)]
    flex_pick = flex_eligible.sort_values('actual_points', ascending=False).head(1)
    starters.append(flex_pick)

    starters_df = pd.concat(starters, ignore_index=True)
    return starters_df['actual_points'].sum()


def score_draft(rosters):
    return {team: score_roster(players) for team, players in rosters.items()}

# Score the smoke-test draft from before.
scores = score_draft(test_rosters)
for team, score in sorted(scores.items(), key=lambda x: -x[1]):
    marker = ' <- our agent' if team == 0 else ''
    print(f"Team {team}: {score:.1f} pts{marker}")


Team 7: 1876.8 pts
Team 2: 1812.0 pts
Team 4: 1742.3 pts
Team 8: 1732.5 pts
Team 0: 1714.2 pts <- our agent
Team 5: 1702.9 pts
Team 1: 1676.9 pts
Team 9: 1621.5 pts
Team 6: 1537.3 pts
Team 3: 1270.0 pts


## Step 5: full evaluation -- every possible draft slot

Since the draft logic is fully deterministic (no real randomness in
tie-breaking), running the same scenario repeatedly wouldn't tell us
anything new. Instead: run the draft once for each of the 10 possible
positions our agent could pick from (1st overall through 10th overall) --
draft slot itself matters a lot, so this gives a genuine range of outcomes
rather than one lucky or unlucky sample.


In [116]:
results = []

for agent_slot in range(N_TEAMS):
    rosters = run_draft(player_pool, agent_team_index=agent_slot)
    scores = score_draft(rosters)

    ranked = sorted(scores.items(), key=lambda x: -x[1])
    agent_rank = [i for i, (team, _) in enumerate(ranked, start=1) if team == agent_slot][0]
    agent_points = scores[agent_slot]
    other_teams_avg = np.mean([s for t, s in scores.items() if t != agent_slot])

    results.append({
        'agent_draft_slot': agent_slot + 1,
        'agent_points': agent_points,
        'agent_rank': agent_rank,
        'other_teams_avg_points': other_teams_avg,
        'won_league': agent_rank == 1,
    })

results_df = pd.DataFrame(results)
results_df


,agent_draft_slot,agent_points,agent_rank,other_teams_avg_points,won_league
0,1,1714.20,5,1663.588889,False
1,2,1689.80,7,1666.344444,False
2,3,1689.80,6,1672.015556,False
3,4,1602.30,7,1703.757778,False
4,5,1641.36,6,1712.793333,False
5,6,1716.64,6,1710.108889,False
6,7,1769.98,4,1683.600000,False
7,8,1648.24,7,1681.775556,False
8,9,1519.06,9,1687.406667,False
9,10,1580.16,9,1686.157778,False


In [117]:
win_rate = results_df['won_league'].mean()
avg_rank = results_df['agent_rank'].mean()
avg_points_diff = (results_df['agent_points'] - results_df['other_teams_avg_points']).mean()

print(f"Win rate (finished #1 of 10): {win_rate:.1%}")
print(f"Average finish rank (1=best, 10=worst): {avg_rank:.1f}")
print(f"Average points vs. average ADP-bot team: {avg_points_diff:+.1f}")
print()
print(f"(For reference: a team with no edge at all would expect a 10% win rate and average rank 5.5)")


Win rate (finished #1 of 10): 0.0%
Average finish rank (1=best, 10=worst): 6.6
Average points vs. average ADP-bot team: -29.6

(For reference: a team with no edge at all would expect a 10% win rate and average rank 5.5)


## Step 6: sanity check -- does a pure-ADP agent land at the expected baseline?

Before reading anything into the previous result, ruling out a simulator
bug: set the agent's model weight to 0 for every position (identical
behavior to an ADP bot) and confirm it lands near the "no edge" expectation
(~5.5 average rank, ~0 point differential). If this control fails, the
issue is in the simulator's mechanics, not the model.


In [118]:
MODEL_WEIGHT_BY_POSITION_CONTROL = {'QB': 0.0, 'RB': 0.0, 'WR': 0.0, 'TE': 0.0}

def compute_agent_scores_control(pool):
    pool = pool.copy()
    for pos in SKILL_POSITIONS:
        pos_mask = pool['position'] == pos
        pos_players = pool[pos_mask]
        model_order = pos_players.sort_values('model_pred', ascending=False)
        adp_order = pos_players.sort_values('adp_avg', ascending=True)['adp_avg'].values
        model_implied_adp = pd.Series(adp_order[:len(model_order)], index=model_order.index)
        pool.loc[model_implied_adp.index, 'model_implied_adp'] = model_implied_adp
    weight_map = pool['position'].map(MODEL_WEIGHT_BY_POSITION_CONTROL)
    pool['agent_score'] = -(weight_map * pool['model_implied_adp'] + (1 - weight_map) * pool['adp_avg'])
    return pool

def run_draft_control(pool, agent_team_index):
    pool = compute_agent_scores_control(pool)
    available = pool.copy()
    rosters = {t: [] for t in range(N_TEAMS)}
    position_counts = {t: {'QB': 0, 'RB': 0, 'WR': 0, 'TE': 0} for t in range(N_TEAMS)}
    pick_order = snake_order(N_TEAMS, TOTAL_ROUNDS)

    for team in pick_order:
        eligible = available[available['position'].map(
            lambda p, t=team: position_counts[t][p] < POSITION_CAPS[p]
        )]
        if eligible.empty:
            eligible = available
        # Both agent and bots use adp_avg here -- true apples-to-apples control.
        pick = eligible.sort_values('adp_avg', ascending=True).iloc[0]
        rosters[team].append(pick)
        position_counts[team][pick['position']] += 1
        available = available[available['player_id'] != pick['player_id']]
    return rosters

control_results = []
for agent_slot in range(N_TEAMS):
    rosters = run_draft_control(player_pool, agent_team_index=agent_slot)
    scores = score_draft(rosters)
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    agent_rank = [i for i, (team, _) in enumerate(ranked, start=1) if team == agent_slot][0]
    control_results.append({'agent_draft_slot': agent_slot + 1, 'agent_rank': agent_rank, 'agent_points': scores[agent_slot]})

control_df = pd.DataFrame(control_results)
print(f"Control (pure ADP vs pure ADP) average rank: {control_df['agent_rank'].mean():.2f} (expect ~5.5)")
control_df


Control (pure ADP vs pure ADP) average rank: 5.50 (expect ~5.5)


,agent_draft_slot,agent_rank,agent_points
0,1,6,1687.94
1,2,7,1675.02
2,3,2,1808.48
3,4,10,1297.28
4,5,8,1666.26
5,6,4,1719.44
6,7,5,1695.98
7,8,1,1810.50
8,9,3,1750.32
9,10,9,1621.52


## Step 7: multi-season evaluation

Testing 2022 and 2023 in addition to 2024, to see if the underperformance
is a consistent pattern or single-season noise. Caveat: 2022-2023 were part
of the model's validation set (used to pick hyperparameters), so they're
not fully clean holdout data for the raw projection model -- but they're
still meaningful here since we're evaluating the DRAFT STRATEGY, a step
removed from the model's MAE-tuning objective.


In [119]:
def build_player_pool(season, features, features_encoded, model, feature_cols, adp_with_outcomes):
    season_df = features_encoded[features_encoded['season'] == season].copy()
    season_df['model_pred'] = model.predict(season_df[feature_cols])

    position_lookup = features[features['season'] == season][['player_id', 'position']]
    adp_season = adp_with_outcomes[adp_with_outcomes['season'] == season][['player_id', 'player_name', 'adp_avg']]

    pool = season_df[['player_id', 'model_pred', 'fantasy_points_ppr']].rename(columns={'fantasy_points_ppr': 'actual_points'})
    pool = pool.merge(position_lookup, on='player_id', how='left')
    pool = pool.merge(adp_season, on='player_id', how='inner')
    return pool.sort_values('adp_avg').reset_index(drop=True)


def evaluate_season(season_pool):
    results = []
    for agent_slot in range(N_TEAMS):
        rosters = run_draft(season_pool, agent_team_index=agent_slot)
        scores = score_draft(rosters)
        ranked = sorted(scores.items(), key=lambda x: -x[1])
        agent_rank = [i for i, (team, _) in enumerate(ranked, start=1) if team == agent_slot][0]
        agent_points = scores[agent_slot]
        other_avg = np.mean([s for t, s in scores.items() if t != agent_slot])
        results.append({'agent_rank': agent_rank, 'agent_points': agent_points, 'other_avg': other_avg,
                         'won_league': agent_rank == 1})
    return pd.DataFrame(results)


all_season_results = {}
for season in [2022, 2023, 2024]:
    pool = build_player_pool(season, features, features_encoded, model, feature_cols, adp_with_outcomes)
    season_results = evaluate_season(pool)
    all_season_results[season] = season_results
    print(f"{season}: win rate {season_results['won_league'].mean():.1%}, "
          f"avg rank {season_results['agent_rank'].mean():.1f}, "
          f"avg pts diff {(season_results['agent_points'] - season_results['other_avg']).mean():+.1f}")


2022: win rate 20.0%, avg rank 3.4, avg pts diff +127.7
2023: win rate 20.0%, avg rank 6.7, avg pts diff -35.3
2024: win rate 0.0%, avg rank 6.6, avg pts diff -29.6


In [120]:
combined = pd.concat(all_season_results.values(), ignore_index=True)
print(f"\nCombined across all 3 seasons (30 simulated drafts):")
print(f"Win rate: {combined['won_league'].mean():.1%}")
print(f"Average rank: {combined['agent_rank'].mean():.2f} (no-edge expectation: 5.5)")
print(f"Average points vs. ADP-bot average: {(combined['agent_points'] - combined['other_avg']).mean():+.1f}")



Combined across all 3 seasons (30 simulated drafts):
Win rate: 13.3%
Average rank: 5.57 (no-edge expectation: 5.5)
Average points vs. ADP-bot average: +20.9


## Step 8: diagnosing the strategy -- draft slot patterns and deviation attribution

**Part A**: does draft slot (1st overall vs. 10th overall) affect our
strategy's results, combined across all 3 seasons?

**Part B (the more useful one)**: for each pick where our agent chose a
DIFFERENT player than a pure-ADP team would have picked from that same
slot, was that swap good or bad in hindsight (using actual_points)? Broken
down by position -- this directly tests whether the RB/TE-weighted
blending is helping or hurting where we intended it to help.


In [121]:
# Part A: draft slot pattern across all 3 seasons combined.
slot_pattern = []
for season, df in all_season_results.items():
    df = df.copy()
    df['season'] = season
    df['draft_slot'] = range(1, N_TEAMS + 1)
    slot_pattern.append(df)

slot_pattern_df = pd.concat(slot_pattern, ignore_index=True)
slot_summary = slot_pattern_df.groupby('draft_slot').agg(
    avg_rank=('agent_rank', 'mean'),
    avg_pts_diff=('agent_points', lambda x: (x - slot_pattern_df.loc[x.index, 'other_avg']).mean()),
    wins=('won_league', 'sum'),
).reset_index()

slot_summary


,draft_slot,avg_rank,avg_pts_diff,wins
0,1,2.666667,174.895556,1
1,2,3.000000,232.333333,2
2,3,5.000000,69.782222,1
3,4,5.666667,8.477037,0
4,5,5.000000,13.993333,0
5,6,5.000000,36.720741,0
6,7,6.000000,-14.711852,0
7,8,6.666667,-49.482963,0
8,9,8.333333,-142.628889,0
9,10,8.333333,-119.960741,0


In [122]:
# Part B: deviation attribution -- compare our agent's actual picks against
# what a pure-ADP team would have picked from the same slot, same season,
# ROUND BY ROUND (both drafts proceed in the same pick order, so comparing
# by round index is the correct "what would I have picked instead" framing).
deviation_records = []

for season in [2022, 2023, 2024]:
    pool = build_player_pool(season, features, features_encoded, model, feature_cols, adp_with_outcomes)
    for agent_slot in range(N_TEAMS):
        agent_rosters = run_draft(pool, agent_team_index=agent_slot)
        control_rosters = run_draft_control(pool, agent_team_index=agent_slot)

        agent_picks = agent_rosters[agent_slot]
        control_picks = control_rosters[agent_slot]

        for rnd in range(len(agent_picks)):
            a_pick, c_pick = agent_picks[rnd], control_picks[rnd]
            if a_pick['player_id'] == c_pick['player_id']:
                continue  # same player picked either way -- not a deviation
            deviation_records.append({
                'season': season,
                'round': rnd + 1,
                'position': a_pick['position'],
                'agent_pick': a_pick['player_name'],
                'agent_pick_points': a_pick['actual_points'],
                'adp_pick': c_pick['player_name'],
                'adp_pick_points': c_pick['actual_points'],
                'swap_value': a_pick['actual_points'] - c_pick['actual_points'],
            })

deviation_df = pd.DataFrame(deviation_records)
print(f"Total deviating picks across all seasons/slots: {len(deviation_df)}")

deviation_summary = deviation_df.groupby('position').agg(
    n=('swap_value', 'count'),
    avg_swap_value=('swap_value', 'mean'),
    pct_positive=('swap_value', lambda x: (x > 0).mean()),
).reset_index()

deviation_summary


Total deviating picks across all seasons/slots: 330


,position,n,avg_swap_value,pct_positive
0,QB,39,61.650256,0.692308
1,RB,125,-5.211680,0.504000
2,TE,48,-51.080000,0.312500
3,WR,118,33.915763,0.635593


## Step 9: retest with revised weights

**Important methodological caveat**: these weights were derived from the
deviation attribution above, which used all 3 seasons INCLUDING 2024. That
means 2024 is no longer a clean, unbiased test for this revised strategy --
we're using its own outcomes to inform the fix. Reporting this honestly as
an in-sample-informed revision, not a fresh validation. A truly clean test
would need a season beyond our current data window.


In [123]:
MODEL_WEIGHT_BY_POSITION = {'QB': 0.55, 'RB': 0.45, 'WR': 0.55, 'TE': 0.10}

all_season_results_v2 = {}
for season in [2022, 2023, 2024]:
    pool = build_player_pool(season, features, features_encoded, model, feature_cols, adp_with_outcomes)
    season_results = evaluate_season(pool)
    all_season_results_v2[season] = season_results
    print(f"{season}: win rate {season_results['won_league'].mean():.1%}, "
          f"avg rank {season_results['agent_rank'].mean():.1f}, "
          f"avg pts diff {(season_results['agent_points'] - season_results['other_avg']).mean():+.1f}")

combined_v2 = pd.concat(all_season_results_v2.values(), ignore_index=True)
print(f"\nCombined (revised weights): win rate {combined_v2['won_league'].mean():.1%}, "
      f"avg rank {combined_v2['agent_rank'].mean():.2f}, "
      f"avg pts diff {(combined_v2['agent_points'] - combined_v2['other_avg']).mean():+.1f}")
print(f"\nOriginal weights for comparison: win rate 13.3%, avg rank 5.57, avg pts diff +20.9")


2022: win rate 0.0%, avg rank 5.5, avg pts diff -25.5
2023: win rate 20.0%, avg rank 4.0, avg pts diff +73.5
2024: win rate 0.0%, avg rank 8.8, avg pts diff -168.5

Combined (revised weights): win rate 6.7%, avg rank 6.10, avg pts diff -40.2

Original weights for comparison: win rate 13.3%, avg rank 5.57, avg pts diff +20.9


## Step 10: is the slot advantage structural, or specific to our strategy?

Running the same slot-by-slot check, but with the pure-ADP-vs-pure-ADP
CONTROL instead of our agent. If snake draft position itself gives an
inherent advantage, we'd expect a similar early-slot-wins pattern here too.
If this looks essentially random by comparison, that confirms the earlier
pattern was really about our strategy having more room to deviate (and
introduce risk) in later rounds -- not about draft position itself.


In [124]:
def evaluate_season_control(season_pool):
    results = []
    for agent_slot in range(N_TEAMS):
        rosters = run_draft_control(season_pool, agent_team_index=agent_slot)
        scores = score_draft(rosters)
        ranked = sorted(scores.items(), key=lambda x: -x[1])
        agent_rank = [i for i, (team, _) in enumerate(ranked, start=1) if team == agent_slot][0]
        agent_points = scores[agent_slot]
        other_avg = np.mean([s for t, s in scores.items() if t != agent_slot])
        results.append({'agent_rank': agent_rank, 'agent_points': agent_points, 'other_avg': other_avg})
    return pd.DataFrame(results)

control_slot_pattern = []
for season in [2022, 2023, 2024]:
    pool = build_player_pool(season, features, features_encoded, model, feature_cols, adp_with_outcomes)
    df = evaluate_season_control(pool)
    df['season'] = season
    df['draft_slot'] = range(1, N_TEAMS + 1)
    control_slot_pattern.append(df)

control_slot_df = pd.concat(control_slot_pattern, ignore_index=True)
control_slot_summary = control_slot_df.groupby('draft_slot').agg(
    avg_rank=('agent_rank', 'mean'),
    avg_pts_diff=('agent_points', lambda x: (x - control_slot_df.loc[x.index, 'other_avg']).mean()),
).reset_index()

print("CONTROL (pure ADP vs pure ADP) by slot:")
print(control_slot_summary)
print()
print("For comparison, OUR AGENT (original weights) by slot was:")
print(slot_summary[['draft_slot', 'avg_rank', 'avg_pts_diff']])


CONTROL (pure ADP vs pure ADP) by slot:
   draft_slot  avg_rank  avg_pts_diff
0           1  6.333333      5.617037
1           2  3.000000    160.172593
2           3  2.666667    123.098519
3           4  8.000000   -199.960741
4           5  8.333333   -101.545926
5           6  3.333333     83.135556
6           7  6.000000    -24.479259
7           8  4.666667     -4.620000
8           9  5.000000     32.394815
9          10  7.666667    -73.812593

For comparison, OUR AGENT (original weights) by slot was:
   draft_slot  avg_rank  avg_pts_diff
0           1  2.666667    174.895556
1           2  3.000000    232.333333
2           3  5.000000     69.782222
3           4  5.666667      8.477037
4           5  5.000000     13.993333
5           6  5.000000     36.720741
6           7  6.000000    -14.711852
7           8  6.666667    -49.482963
8           9  8.333333   -142.628889
9          10  8.333333   -119.960741


## Step 11: which slot actually wins, under identical strategy for everyone?

Under pure ADP for all 10 teams, the draft outcome doesn't depend on which
team is "labeled" the agent -- everyone behaves identically, so there's
only ONE unique draft per season here. Running it once per season and
checking which slot number produced the winning (and losing) team directly
answers: is there a real structural draft-position advantage, independent
of any strategy?


In [125]:
for season in [2022, 2023, 2024]:
    pool = build_player_pool(season, features, features_encoded, model, feature_cols, adp_with_outcomes)
    rosters = run_draft_control(pool, agent_team_index=0)  # label doesn't matter, all teams identical
    scores = score_draft(rosters)

    ranked = sorted(scores.items(), key=lambda x: -x[1])
    print(f"--- {season} (pure ADP for all 10 teams) ---")
    for rank, (slot, score) in enumerate(ranked, start=1):
        print(f"  Rank {rank}: draft slot {slot + 1}, {score:.1f} pts")
    print()


--- 2022 (pure ADP for all 10 teams) ---
  Rank 1: draft slot 2, 1877.7 pts
  Rank 2: draft slot 3, 1808.6 pts
  Rank 3: draft slot 8, 1751.3 pts
  Rank 4: draft slot 6, 1673.2 pts
  Rank 5: draft slot 10, 1667.7 pts
  Rank 6: draft slot 4, 1637.4 pts
  Rank 7: draft slot 7, 1634.5 pts
  Rank 8: draft slot 1, 1624.9 pts
  Rank 9: draft slot 9, 1559.1 pts
  Rank 10: draft slot 5, 1518.8 pts

--- 2023 (pure ADP for all 10 teams) ---
  Rank 1: draft slot 2, 1843.5 pts
  Rank 2: draft slot 6, 1795.6 pts
  Rank 3: draft slot 9, 1741.8 pts
  Rank 4: draft slot 3, 1679.1 pts
  Rank 5: draft slot 1, 1666.1 pts
  Rank 6: draft slot 7, 1567.2 pts
  Rank 7: draft slot 5, 1504.5 pts
  Rank 8: draft slot 4, 1489.2 pts
  Rank 9: draft slot 10, 1475.3 pts
  Rank 10: draft slot 8, 1389.5 pts

--- 2024 (pure ADP for all 10 teams) ---
  Rank 1: draft slot 8, 1810.5 pts
  Rank 2: draft slot 3, 1808.5 pts
  Rank 3: draft slot 9, 1750.3 pts
  Rank 4: draft slot 6, 1719.4 pts
  Rank 5: draft slot 7, 1696.0 

## Step 12: named draft strategy tournament

Testing real, named fantasy draft strategies against each other in the same
draft (not just "1 agent vs. 9 identical ADP bots"). Each strategy is
defined as forced/banned positions per round, otherwise best-available-by-
ADP. To control for draft slot advantage (Step 11 showed it's real), each
strategy's slot assignment is randomized across multiple runs and averaged.


In [126]:
# Reset to the ORIGINAL weights -- Step 9's revised weights performed worse
# overall and were not kept as final. The global was left in the revised
# state from that experiment, so resetting explicitly here.
MODEL_WEIGHT_BY_POSITION = {'QB': 0.30, 'RB': 0.55, 'WR': 0.35, 'TE': 0.55}

STRATEGIES = {
    'Pure ADP': {'forced': {}, 'banned': {}},
    'Zero RB': {'forced': {}, 'banned': {r: {'RB'} for r in range(1, 6)}},
    'Hero RB': {'forced': {1: 'RB'}, 'banned': {r: {'RB'} for r in range(2, 6)}},
    'Robust RB': {'forced': {1: 'RB', 2: 'RB', 3: 'RB'}, 'banned': {}},
    'Elite TE': {'forced': {2: 'TE'}, 'banned': {}},
    'Punt QB/TE': {'forced': {}, 'banned': {r: {'QB', 'TE'} for r in range(1, 8)}},
    'Model Blend': {'forced': {}, 'banned': {}, 'use_model': True},
}


def rule_based_pick(eligible, round_num, strategy):
    forced_pos = strategy['forced'].get(round_num)
    banned_pos = strategy['banned'].get(round_num, set())

    candidates = eligible[~eligible['position'].isin(banned_pos)]
    if forced_pos and (candidates['position'] == forced_pos).any():
        candidates = candidates[candidates['position'] == forced_pos]
    if candidates.empty:
        candidates = eligible  # fallback if the forced/banned rule leaves nothing

    if strategy.get('use_model'):
        return candidates.sort_values('agent_score', ascending=False).iloc[0]
    return candidates.sort_values('adp_avg', ascending=True).iloc[0]


def run_strategy_draft(pool, slot_assignments):
    """slot_assignments: dict of {team_index: strategy_name}"""
    pool_scored = compute_agent_scores(pool)  # needed for the Model Blend strategy's agent_score
    available = pool_scored.copy()
    rosters = {t: [] for t in range(N_TEAMS)}
    position_counts = {t: {'QB': 0, 'RB': 0, 'WR': 0, 'TE': 0} for t in range(N_TEAMS)}
    pick_order = snake_order(N_TEAMS, TOTAL_ROUNDS)

    round_counter = {t: 0 for t in range(N_TEAMS)}
    for team in pick_order:
        round_counter[team] += 1
        strategy = STRATEGIES[slot_assignments[team]]

        eligible = available[available['position'].map(
            lambda p, t=team: position_counts[t][p] < POSITION_CAPS[p]
        )]
        if eligible.empty:
            eligible = available

        pick = rule_based_pick(eligible, round_counter[team], strategy)
        rosters[team].append(pick)
        position_counts[team][pick['position']] += 1
        available = available[available['player_id'] != pick['player_id']]

    return rosters

print(f"Strategies defined: {list(STRATEGIES.keys())}")


Strategies defined: ['Pure ADP', 'Zero RB', 'Hero RB', 'Robust RB', 'Elite TE', 'Punt QB/TE', 'Model Blend']


In [127]:
random.seed(42)
strategy_names = list(STRATEGIES.keys())  # 7 strategies
filler_count = N_TEAMS - len(strategy_names)  # 3 filler slots
N_SLOT_ROTATIONS = 8  # how many random slot assignments to average over, per season

tournament_records = []

for season in [2022, 2023, 2024]:
    pool = build_player_pool(season, features, features_encoded, model, feature_cols, adp_with_outcomes)

    for rotation in range(N_SLOT_ROTATIONS):
        all_slot_strategies = strategy_names + ['Pure ADP'] * filler_count
        random.shuffle(all_slot_strategies)
        slot_assignments = {i: s for i, s in enumerate(all_slot_strategies)}

        rosters = run_strategy_draft(pool, slot_assignments)
        scores = score_draft(rosters)
        ranked = sorted(scores.items(), key=lambda x: -x[1])
        rank_lookup = {team: rank for rank, (team, _) in enumerate(ranked, start=1)}

        for team, strat_name in slot_assignments.items():
            tournament_records.append({
                'season': season,
                'rotation': rotation,
                'draft_slot': team + 1,
                'strategy': strat_name,
                'points': scores[team],
                'rank': rank_lookup[team],
                'won': rank_lookup[team] == 1,
            })

tournament_df = pd.DataFrame(tournament_records)
print(f"Total team-drafts simulated: {len(tournament_df)}")


Total team-drafts simulated: 240


In [128]:
strategy_summary = tournament_df.groupby('strategy').agg(
    n=('rank', 'count'),
    avg_rank=('rank', 'mean'),
    avg_points=('points', 'mean'),
    win_rate=('won', 'mean'),
    points_std=('points', 'std'),  # consistency -- lower = more consistent
).sort_values('avg_rank').reset_index()

strategy_summary


,strategy,n,avg_rank,avg_points,win_rate,points_std
0,Robust RB,24,4.208333,1736.288333,0.250000,151.859098
1,Zero RB,24,5.208333,1662.103333,0.041667,116.500475
2,Hero RB,24,5.291667,1683.290000,0.125000,137.744461
3,Elite TE,24,5.458333,1659.244167,0.083333,122.722592
4,Pure ADP,96,5.718750,1657.115000,0.083333,131.508772
5,Punt QB/TE,24,5.833333,1622.237500,0.083333,147.387836
6,Model Blend,24,6.125000,1660.056667,0.083333,122.398318


## Step 13: does Robust RB hold up consistently, or is it one lucky pattern?

Checking two things: does it win across ALL 3 seasons (not just one), and
does it work regardless of which draft slot it lands in (not just a few
lucky slots)?


In [129]:
per_season = tournament_df.groupby(['season', 'strategy']).agg(
    avg_rank=('rank', 'mean'),
    win_rate=('won', 'mean'),
    n=('rank', 'count'),
).reset_index()

print("Robust RB by season:")
print(per_season[per_season['strategy'] == 'Robust RB'].to_string(index=False))
print()
print("For comparison, Model Blend by season:")
print(per_season[per_season['strategy'] == 'Model Blend'].to_string(index=False))


Robust RB by season:
 season  strategy  avg_rank  win_rate  n
   2022 Robust RB     4.875     0.000  8
   2023 Robust RB     4.625     0.375  8
   2024 Robust RB     3.125     0.375  8

For comparison, Model Blend by season:
 season    strategy  avg_rank  win_rate  n
   2022 Model Blend     5.000      0.25  8
   2023 Model Blend     7.125      0.00  8
   2024 Model Blend     6.250      0.00  8


In [130]:
robust_rb_by_slot = tournament_df[tournament_df['strategy'] == 'Robust RB'].sort_values('draft_slot')
print(f"Robust RB across {len(robust_rb_by_slot)} team-drafts, by draft slot:")
robust_rb_by_slot[['season', 'rotation', 'draft_slot', 'rank', 'points', 'won']]


Robust RB across 24 team-drafts, by draft slot:


,season,rotation,draft_slot,rank,points,won
10,2022,1,1,7,1614.08,False
210,2024,5,1,3,1797.22,False
150,2023,7,1,1,1935.54,True
140,2023,6,1,1,1912.50,True
120,2023,4,1,1,1852.70,True
1,2022,0,2,3,1774.74,False
71,2022,7,2,3,1786.74,False
181,2024,2,2,1,2022.28,True
172,2024,1,3,10,1611.60,False
52,2022,5,3,4,1664.98,False


In [131]:
print(f"Robust RB win rate by draft slot (n per slot will be small/uneven -- slots were randomly assigned):")
print(robust_rb_by_slot.groupby('draft_slot').agg(avg_rank=('rank', 'mean'), win_rate=('won', 'mean'), n=('rank', 'count')))


Robust RB win rate by draft slot (n per slot will be small/uneven -- slots were randomly assigned):
            avg_rank  win_rate  n
draft_slot                       
1           2.600000  0.600000  5
2           2.333333  0.333333  3
3           5.666667  0.000000  3
4           5.000000  0.000000  1
5           2.000000  0.000000  1
7           4.000000  0.500000  2
8           4.000000  0.250000  4
9           7.666667  0.000000  3
10          5.000000  0.000000  2
